# 02 — Quantization Analysis: PTQ vs QAT

Comparativa cuantitativa de la degradación geométrica bajo PTQ y QAT.
Ejecuta los 5 bloques de evaluación y visualiza los resultados en tablas y gráficos.

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from geoquant.data.dataset import get_dataloaders
from geoquant.evaluation.suite import EvaluationSuite
from geoquant.evaluation.reporter import Reporter
from geoquant.evaluation.embeddings import extract_and_save, load_embeddings
from geoquant.utils.reproducibility import seed_everything

seed_everything(42)

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cpu')
_, val_loader = get_dataloaders(config)

In [ ]:
CHECKPOINTS = {
    'fp32': '../outputs/checkpoints/best_fp32.pth',
    'ptq':  '../outputs/checkpoints/mobilenet_v3_small_ptq_int8.pth',
    'qat':  '../outputs/checkpoints/mobilenet_v3_small_qat_int8.pth',
}
EMB_DIR = Path('../outputs/embeddings')

def load_or_extract(key):
    """Carga embeddings desde disco; extrae y guarda si no existen."""
    emb_path = EMB_DIR / f'emb_{key}.pt'
    if emb_path.exists():
        print(f'[{key}] Cargando desde disco...')
        return load_embeddings(emb_path)
    print(f'[{key}] Extrayendo embeddings...')
    return extract_and_save(
        ckpt_path=CHECKPOINTS[key],
        output_path=emb_path,
        config=config,
        dataloader=val_loader,
        device=device,
    )

emb_fp32, labels = load_or_extract('fp32')
emb_ptq, _       = load_or_extract('ptq')
emb_qat, _       = load_or_extract('qat')

print(f'\nFP32 : {emb_fp32.shape}')
print(f'PTQ  : {emb_ptq.shape}')
print(f'QAT  : {emb_qat.shape}')

In [ ]:
suite = EvaluationSuite(k_neighbors=10)
results_ptq = suite.run(emb_fp32, emb_ptq, labels)
results_qat = suite.run(emb_fp32, emb_qat, labels)

reporter = Reporter('../outputs/results')
reporter.save_all(results_ptq, stem='eval_ptq')
reporter.save_all(results_qat, stem='eval_qat')

print('Resultados guardados en outputs/results/')